# ETL RAW → Silver

Projeto: **Mapa da Cidadania e Acesso à Informação no DF**.

Este notebook limpa, padroniza e integra a camada RAW na camada Silver. A tabela principal gerada tem granularidade **1 Região Administrativa + 1 ano**. Como os arquivos LAI disponíveis não trazem Região Administrativa, a LAI é agregada separadamente por órgão e ano.

In [1]:

from pathlib import Path
import re
import unicodedata
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Transformer":
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "Data Layer" / "raw"
SILVER_DIR = PROJECT_ROOT / "Data Layer" / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)
ANOS_PROJETO = [2023, 2024, 2025]
CODIGOS_ESPECIAIS_PDAD = {77777, 88888, 99999, 77777.0, 88888.0, 99999.0, "77777", "88888", "99999"}

def remover_acentos(texto):
    if pd.isna(texto): return texto
    return "".join(c for c in unicodedata.normalize("NFKD", str(texto)) if not unicodedata.combining(c))

def padronizar_colunas(df):
    cols=[]; usados={}
    for col in df.columns:
        novo=remover_acentos(col).lower().strip().replace('.', '_')
        novo=re.sub(r"\s+", "_", novo)
        novo=re.sub(r"[^a-z0-9_]+", "_", novo)
        novo=re.sub(r"_+", "_", novo).strip('_') or 'coluna'
        base=novo
        if base in usados:
            usados[base]+=1; novo=f"{base}_{usados[base]}"
        else:
            usados[base]=0
        cols.append(novo)
    out=df.copy(); out.columns=cols; return out

def normalizar_chave(texto):
    if pd.isna(texto): return np.nan
    texto=remover_acentos(texto).lower().strip()
    texto=re.sub(r"[^a-z0-9]+", " ", texto)
    texto=re.sub(r"\s+", " ", texto).strip()
    aliases_ra = {
        'sudoeste octogonal': 'sudoeste e octogonal',
        'sol nascente por do sol': 'sol nascente por do sol',
    }
    return aliases_ra.get(texto, texto)

def normalizar_nome_orgao(texto):
    if pd.isna(texto) or str(texto).strip()=="": return "Não informado"
    return re.sub(r"\s+", " ", str(texto)).strip()

def ler_csv_auto(caminho, nrows=None):
    melhor=None; erros=[]
    for kwargs in [{"sep":";","encoding":"utf-8-sig"},{"sep":";","encoding":"latin1"},{"sep":",","encoding":"utf-8-sig"},{"sep":",","encoding":"latin1"}]:
        try:
            df=pd.read_csv(caminho, nrows=nrows, **kwargs)
            if melhor is None or len(df.columns)>len(melhor.columns): melhor=df
            if len(df.columns)>1: return df
        except Exception as exc:
            erros.append(f"{kwargs}: {exc}")
    if melhor is not None: return melhor
    raise RuntimeError("Não foi possível ler CSV: "+" | ".join(erros))

def converter_colunas_data(df):
    out=df.copy()
    for col in [c for c in out.columns if c.startswith('dt_') or '_dt_' in c or 'data' in c]:
        out[col]=pd.to_datetime(out[col], dayfirst=True, errors='coerce')
    return out

def para_numero(serie):
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors='coerce')
    s=serie.astype(str).str.strip().replace({'nan':np.nan,'None':np.nan,'':np.nan})
    tem_virgula=s.str.contains(',', na=False)
    s=s.mask(tem_virgula, s[tem_virgula].str.replace('.', '', regex=False).str.replace(',', '.', regex=False))
    return pd.to_numeric(s, errors='coerce')

def tratar_codigos_especiais(df, preservar=None):
    preservar=set(preservar or []); out=df.copy()
    for col in out.columns:
        if col in preservar: continue
        if pd.api.types.is_numeric_dtype(out[col]): out[col]=out[col].replace(list(CODIGOS_ESPECIAIS_PDAD), np.nan)
        else: out[col]=out[col].replace(['77777','88888','99999'], np.nan)
    return out

def media_ponderada(valor, peso):
    dados=pd.DataFrame({'valor':valor,'peso':peso}).dropna(); dados=dados[dados['peso']>0]
    return np.nan if dados.empty or dados['peso'].sum()==0 else np.average(dados['valor'], weights=dados['peso'])

def percentual_ponderado(condicao, peso):
    dados=pd.DataFrame({'condicao':condicao,'peso':peso}).dropna(); dados=dados[dados['peso']>0]
    return np.nan if dados.empty or dados['peso'].sum()==0 else 100*dados.loc[dados['condicao'].astype(bool),'peso'].sum()/dados['peso'].sum()

def salvar_csv(df, nome):
    caminho=SILVER_DIR/nome
    df.to_csv(caminho, index=False, encoding='utf-8-sig')
    print(f"Salvo: {caminho.relative_to(PROJECT_ROOT)} | linhas={len(df):,} colunas={len(df.columns):,}")
    return caminho

# Dicionário PDAD
anexo_localidade=padronizar_colunas(pd.read_excel(RAW_DIR/'dicionario_de_variaveis_pdada_2024_público.xlsx', sheet_name='anexo_1')).ffill()
anexo_localidade['valor']=pd.to_numeric(anexo_localidade['valor'], errors='coerce')
mapa_localidade=dict(zip(anexo_localidade['valor'], anexo_localidade['descricao_do_valor']))

# LAI pedidos
pedidos=converter_colunas_data(padronizar_colunas(ler_csv_auto(RAW_DIR/'Participa DF _LAI_ - Pedidos.csv')))
pedidos['ano']=pd.to_numeric(pedidos['nr_ano_pedido'], errors='coerce').astype('Int64') if 'nr_ano_pedido' in pedidos.columns else pedidos.get('dt_pedido', pd.Series(pd.NaT,index=pedidos.index)).dt.year.astype('Int64')
col_data_pedido='dt_pedido' if 'dt_pedido' in pedidos.columns else 'dt_data_pedido'
if col_data_pedido in pedidos.columns:
    pedidos['mes']=pedidos[col_data_pedido].dt.month.astype('Int64'); pedidos['ano_mes']=pedidos[col_data_pedido].dt.to_period('M').astype(str)
else:
    pedidos['mes']=pd.NA; pedidos['ano_mes']=pedidos['ano'].astype(str)
col_resposta='respostas_dt_resposta'
pedidos['dias_resposta']=((pedidos[col_resposta]-pedidos[col_data_pedido]).dt.total_seconds()/86400) if col_data_pedido in pedidos.columns and col_resposta in pedidos.columns else np.nan
pedidos['pedido_sem_resposta']=pedidos.get(col_resposta, pd.Series(pd.NaT,index=pedidos.index)).isna()
if 'ds_unidade' in pedidos.columns: pedidos['ds_unidade']=pedidos['ds_unidade'].map(normalizar_nome_orgao)
for col in [c for c in pedidos.columns if 'classificacao' in c or 'tipo_resposta' in c or 'situacao' in c]: pedidos[col]=pedidos[col].fillna('Não informado')
pedidos=pedidos[pedidos['ano'].isin(ANOS_PROJETO)].copy()
salvar_csv(pedidos, 'tb_lai_pedidos_silver.csv')

# LAI recursos
recursos=converter_colunas_data(padronizar_colunas(ler_csv_auto(RAW_DIR/'Participa DF _LAI_ - Recursos.csv')))
if 'dt_recurso' in recursos.columns:
    recursos['ano']=recursos['dt_recurso'].dt.year.astype('Int64'); recursos['mes']=recursos['dt_recurso'].dt.month.astype('Int64'); recursos['ano_mes']=recursos['dt_recurso'].dt.to_period('M').astype(str)
else:
    recursos['ano']=pd.NA; recursos['mes']=pd.NA; recursos['ano_mes']=pd.NA
recursos=recursos[recursos['ano'].isin(ANOS_PROJETO)].drop_duplicates().copy()
for col in [c for c in recursos.columns if 'classificacao' in c or 'tipo_resposta' in c or 'instancia' in c]: recursos[col]=recursos[col].fillna('Não informado')
salvar_csv(recursos, 'tb_lai_recursos_silver.csv')

# LAI satisfação
satisfacao=converter_colunas_data(padronizar_colunas(ler_csv_auto(RAW_DIR/'Participa DF _LAI_ - Pesquisa de Satisfação.csv')))
col_data_satisfacao=next((c for c in satisfacao.columns if c.startswith('dt_') or 'data' in c), None)
if col_data_satisfacao:
    satisfacao['ano']=satisfacao[col_data_satisfacao].dt.year.astype('Int64'); satisfacao['mes']=satisfacao[col_data_satisfacao].dt.month.astype('Int64'); satisfacao['ano_mes']=satisfacao[col_data_satisfacao].dt.to_period('M').astype(str)
    satisfacao=satisfacao[satisfacao['ano'].isin(ANOS_PROJETO)].copy()
else:
    satisfacao['ano']=pd.NA; satisfacao['mes']=pd.NA; satisfacao['ano_mes']=pd.NA
for col in satisfacao.select_dtypes(include='object').columns: satisfacao[col]=satisfacao[col].fillna('Não informado')
salvar_csv(satisfacao, 'tb_lai_satisfacao_silver.csv')

# PDAD moradores
moradores=padronizar_colunas(ler_csv_auto(RAW_DIR/'moradores.csv'))
if 'peso_mor' in moradores.columns: moradores['peso_mor']=para_numero(moradores['peso_mor'])
for col in ['idade_calculada','renda_ind','renda_ind_r','escolaridade','localidade']:
    if col in moradores.columns: moradores[col]=para_numero(moradores[col])
moradores=tratar_codigos_especiais(moradores, preservar={'localidade','setor_distrito','a01uf','a01nficha','morador_id','index'})
if 'localidade' in moradores.columns:
    moradores['regiao_administrativa']=moradores['localidade'].map(mapa_localidade).fillna(moradores['localidade'].astype(str))
    moradores['ra_chave']=moradores['regiao_administrativa'].map(normalizar_chave)
salvar_csv(moradores, 'tb_pdad_moradores_silver.csv')

# PDAD domicílios
domicilios=padronizar_colunas(pd.read_excel(RAW_DIR/'domicilios.xlsx'))
if 'peso_dom' in domicilios.columns: domicilios['peso_dom']=para_numero(domicilios['peso_dom'])
for col in domicilios.columns:
    if col!='starttime' and domicilios[col].dtype=='object':
        convertido=para_numero(domicilios[col])
        if convertido.notna().mean()>0.8: domicilios[col]=convertido
for col in ['localidade','a01npessoas','b03','c05','renda_domiciliar','renda_domiciliar_r','renda_domiciliar_pc','renda_domiciliar_pc_r']:
    if col in domicilios.columns: domicilios[col]=para_numero(domicilios[col])
domicilios=tratar_codigos_especiais(domicilios, preservar={'localidade','setor_distrito','a01uf','a01nficha'})
if 'localidade' in domicilios.columns:
    domicilios['regiao_administrativa']=domicilios['localidade'].map(mapa_localidade).fillna(domicilios['localidade'].astype(str))
    domicilios['ra_chave']=domicilios['regiao_administrativa'].map(normalizar_chave)
salvar_csv(domicilios, 'tb_pdad_domicilios_silver.csv')

# Projeções populacionais
def classificar_faixa_etaria(valor):
    texto=str(valor).strip().lower()
    if texto in {'nan', 'total'}:
        return np.nan
    if '+' in texto:
        inicio=pd.to_numeric(re.findall(r'\d+', texto)[0], errors='coerce')
    else:
        nums=re.findall(r'\d+', texto)
        inicio=pd.to_numeric(nums[0], errors='coerce') if nums else np.nan
    if pd.isna(inicio):
        return np.nan
    if inicio <= 14:
        return '0-14'
    if inicio <= 59:
        return '15-59'
    return '60+'

def extrair_populacao_sheet(caminho, sheet_name):
    bruto=pd.read_excel(caminho, sheet_name=sheet_name, header=None)
    anos=bruto.iloc[3].ffill(); sexos=bruto.iloc[4]
    dados=bruto.iloc[5:].copy().rename(columns={0:'idade'})
    dados['faixa_etaria']=dados['idade'].map(classificar_faixa_etaria)
    dados=dados[dados['faixa_etaria'].notna()].copy()
    partes=[]
    for col in dados.columns[1:]:
        if col == 'faixa_etaria':
            continue
        ano=pd.to_numeric(anos[col], errors='coerce'); sexo=sexos[col]
        if pd.isna(ano) or pd.isna(sexo): continue
        sexo=str(sexo).strip()
        if sexo not in {'Total','Homens','Mulheres'}: continue
        tmp=dados[['faixa_etaria',col]].rename(columns={col:'populacao'})
        tmp['ano']=int(ano); tmp['sexo']=sexo; partes.append(tmp)
    if not partes: return pd.DataFrame(columns=['regiao_administrativa','ano','sexo','faixa_etaria','populacao'])
    longo=pd.concat(partes, ignore_index=True); longo['populacao']=pd.to_numeric(longo['populacao'], errors='coerce')
    longo['regiao_administrativa']=sheet_name
    return longo[['regiao_administrativa','ano','sexo','faixa_etaria','populacao']]

pop_path=RAW_DIR/'Dados - Projeções populacionais por Região Administrativa, sexo e faixa etária 2020-2030_1.xlsx'
xl_pop=pd.ExcelFile(pop_path); sheets_ra=[s for s in xl_pop.sheet_names if s not in {'DF', "RA's"}]
pop_longa=pd.concat([extrair_populacao_sheet(pop_path, s) for s in sheets_ra], ignore_index=True)
pop_longa=pop_longa[pop_longa['ano'].isin(ANOS_PROJETO)].copy(); pop_longa['ra_chave']=pop_longa['regiao_administrativa'].map(normalizar_chave)
pop_total=pop_longa[pop_longa['sexo']=='Total'].pivot_table(index=['regiao_administrativa','ra_chave','ano'], columns='faixa_etaria', values='populacao', aggfunc='sum').reset_index().rename(columns={'0-14':'populacao_0_14','15-59':'populacao_15_59','60+':'populacao_60_mais','60':'populacao_60_mais'})
for col in ['populacao_0_14','populacao_15_59','populacao_60_mais']:
    if col not in pop_total.columns:
        pop_total[col]=0
pop_sexo=pop_longa[pop_longa['sexo'].isin(['Homens','Mulheres'])].groupby(['regiao_administrativa','ra_chave','ano','sexo'], as_index=False)['populacao'].sum().pivot_table(index=['regiao_administrativa','ra_chave','ano'], columns='sexo', values='populacao', aggfunc='sum').reset_index().rename(columns={'Homens':'populacao_masculina','Mulheres':'populacao_feminina'})
for col in ['populacao_masculina','populacao_feminina']:
    if col not in pop_sexo.columns:
        pop_sexo[col]=np.nan
pop_ra_ano=pop_total.merge(pop_sexo, on=['regiao_administrativa','ra_chave','ano'], how='left')
pop_ra_ano['populacao_total']=pop_ra_ano[['populacao_0_14','populacao_15_59','populacao_60_mais']].sum(axis=1)
pop_ra_ano=pop_ra_ano[['ano','regiao_administrativa','ra_chave','populacao_total','populacao_masculina','populacao_feminina','populacao_0_14','populacao_15_59','populacao_60_mais']].sort_values(['regiao_administrativa','ano']).reset_index(drop=True)
salvar_csv(pop_ra_ano, 'tb_populacao_ra_ano_silver.csv')

# Indicadores PDAD ponderados
indic_mor=[]
for chave, grupo in moradores.groupby('ra_chave', dropna=True):
    indic_mor.append({'ra_chave':chave,'renda_media_ponderada':media_ponderada(grupo.get('renda_ind_r', grupo.get('renda_ind')), grupo['peso_mor']),'idade_media_ponderada':media_ponderada(grupo.get('idade_calculada'), grupo['peso_mor']),'percentual_baixa_escolaridade':percentual_ponderado(grupo.get('escolaridade').isin([1,2,3]), grupo['peso_mor'])})
indic_mor=pd.DataFrame(indic_mor)
indic_dom=[]
for chave, grupo in domicilios.groupby('ra_chave', dropna=True):
    indic_dom.append({'ra_chave':chave,'media_moradores_por_domicilio':media_ponderada(grupo.get('a01npessoas'), grupo['peso_dom']),'percentual_domicilios_com_internet':percentual_ponderado(grupo.get('c05').isin([1,2]), grupo['peso_dom']),'percentual_domicilios_proprios':percentual_ponderado(grupo.get('b03').isin([1,2]), grupo['peso_dom']),'percentual_domicilios_alugados':percentual_ponderado(grupo.get('b03').eq(3), grupo['peso_dom'])})
indic_dom=pd.DataFrame(indic_dom)
indic_pdad=indic_mor.merge(indic_dom, on='ra_chave', how='outer')

# LAI por órgão e ano
class_col='respostas_ds_tipo_classificacao_resposta' if 'respostas_ds_tipo_classificacao_resposta' in pedidos.columns else None
org_col='ds_unidade' if 'ds_unidade' in pedidos.columns else None
if org_col:
    lai_orgao=pedidos.groupby([org_col,'ano'], dropna=False).agg(qtd_pedidos_lai=('nr_pedido_participa','nunique'), tempo_medio_resposta_dias=('dias_resposta','mean'), percentual_sem_resposta=('pedido_sem_resposta', lambda s: 100*s.mean())).reset_index().rename(columns={org_col:'orgao'})
else:
    lai_orgao=pedidos.groupby(['ano'], dropna=False).agg(qtd_pedidos_lai=('nr_pedido_participa','nunique'), tempo_medio_resposta_dias=('dias_resposta','mean'), percentual_sem_resposta=('pedido_sem_resposta', lambda s: 100*s.mean())).reset_index(); lai_orgao['orgao']='Não informado'
if class_col and org_col:
    def pct_class(s, termo): return 100*s.fillna('Não informado').str.contains(termo, case=False, na=False).mean()
    extra=pedidos.groupby([org_col,'ano'], dropna=False)[class_col].agg(percentual_acesso_concedido=lambda s:pct_class(s, r'Acesso Concedido$|^Acesso Concedido'), percentual_acesso_negado=lambda s:pct_class(s, r'Acesso Negado'), percentual_acesso_parcial=lambda s:pct_class(s, r'Parcial')).reset_index().rename(columns={org_col:'orgao'})
    lai_orgao=lai_orgao.merge(extra, on=['orgao','ano'], how='left')
rec_por_pedido=recursos.groupby(['nr_pedido_participa','ano'], as_index=False).size().rename(columns={'size':'qtd_recursos_lai'})
rec_com_orgao=rec_por_pedido.merge(pedidos[['nr_pedido_participa','ano',org_col]].drop_duplicates() if org_col else pedidos[['nr_pedido_participa','ano']].drop_duplicates(), on=['nr_pedido_participa','ano'], how='left')
if org_col:
    rec_orgao=rec_com_orgao.groupby([org_col,'ano'], dropna=False)['qtd_recursos_lai'].sum().reset_index().rename(columns={org_col:'orgao'})
else:
    rec_orgao=rec_com_orgao.groupby(['ano'], dropna=False)['qtd_recursos_lai'].sum().reset_index(); rec_orgao['orgao']='Não informado'
lai_orgao=lai_orgao.merge(rec_orgao, on=['orgao','ano'], how='left')
lai_orgao['qtd_recursos_lai']=lai_orgao['qtd_recursos_lai'].fillna(0).astype(int)
lai_orgao['taxa_recurso_lai']=np.where(lai_orgao['qtd_pedidos_lai']>0, lai_orgao['qtd_recursos_lai']/lai_orgao['qtd_pedidos_lai'], np.nan)
lai_orgao=lai_orgao.sort_values(['ano','orgao']).reset_index(drop=True)
salvar_csv(lai_orgao, 'tb_lai_orgao_ano_silver.csv')

# Consolidada RA/ano, sem LAI territorial artificial
possiveis_territorio=['regiao','regiao_administrativa','localidade','cidade','ra','territorio','endereco']
colunas_territoriais_lai=[c for c in pedidos.columns if any(t in c for t in possiveis_territorio)]
print('Colunas territoriais LAI encontradas:', colunas_territoriais_lai)
mapa_cidadania=pop_ra_ano.merge(indic_pdad, on='ra_chave', how='left').drop(columns=['ra_chave'])
ordem=['ano','regiao_administrativa','populacao_total','populacao_masculina','populacao_feminina','populacao_0_14','populacao_15_59','populacao_60_mais','renda_media_ponderada','idade_media_ponderada','media_moradores_por_domicilio','percentual_baixa_escolaridade','percentual_domicilios_com_internet','percentual_domicilios_proprios','percentual_domicilios_alugados']
mapa_cidadania=mapa_cidadania[[c for c in ordem if c in mapa_cidadania.columns]].sort_values(['regiao_administrativa','ano']).reset_index(drop=True)
salvar_csv(mapa_cidadania, 'tb_mapa_cidadania_ra_ano_silver.csv')

# Validações
arquivos_esperados=['tb_pdad_moradores_silver.csv','tb_pdad_domicilios_silver.csv','tb_lai_pedidos_silver.csv','tb_lai_recursos_silver.csv','tb_lai_satisfacao_silver.csv','tb_populacao_ra_ano_silver.csv','tb_mapa_cidadania_ra_ano_silver.csv','tb_lai_orgao_ano_silver.csv']
for nome in arquivos_esperados: assert (SILVER_DIR/nome).exists(), f'Arquivo não gerado: {nome}'
assert not mapa_cidadania.duplicated(['regiao_administrativa','ano']).any(), 'Duplicidade na granularidade RA/ano'
assert set(mapa_cidadania['ano'].dropna().astype(int).unique()).issubset(set(ANOS_PROJETO))
assert set(pedidos['ano'].dropna().astype(int).unique()).issubset(set(ANOS_PROJETO))
assert set(recursos['ano'].dropna().astype(int).unique()).issubset(set(ANOS_PROJETO))
resumo=pd.DataFrame({'arquivo':arquivos_esperados,'linhas':[len(pd.read_csv(SILVER_DIR/nome)) for nome in arquivos_esperados]})
display(resumo)


Salvo: Data Layer/silver/tb_lai_pedidos_silver.csv | linhas=70,946 colunas=26
Salvo: Data Layer/silver/tb_lai_recursos_silver.csv | linhas=3,998 colunas=14
Salvo: Data Layer/silver/tb_lai_satisfacao_silver.csv | linhas=7,369 colunas=7


/tmp/ipykernel_472832/4007278850.py:139: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in satisfacao.select_dtypes(include='object').columns: satisfacao[col]=satisfacao[col].fillna('Não informado')


/tmp/ipykernel_472832/4007278850.py:56: DtypeWarning: Columns (0: E03_1_1, 1: E03_2_1) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(caminho, nrows=nrows, **kwargs)


/tmp/ipykernel_472832/4007278850.py:149: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  moradores['regiao_administrativa']=moradores['localidade'].map(mapa_localidade).fillna(moradores['localidade'].astype(str))
/tmp/ipykernel_472832/4007278850.py:150: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  moradores['ra_chave']=moradores['regiao_administrativa'].map(normalizar_chave)


Salvo: Data Layer/silver/tb_pdad_moradores_silver.csv | linhas=69,542 colunas=136


...

Salvo: Data Layer/silver/tb_populacao_ra_ano_silver.csv | linhas=99 colunas=9


Salvo: Data Layer/silver/tb_lai_orgao_ano_silver.csv | linhas=288 colunas=10
Colunas territoriais LAI encontradas: ['grau_sigilo_n1', 'grau_sigilo_n2', 'ds_tipo_entrada', 'dt_inicio_prazo', 'dt_termino_prazo', 'dt_max_prazo_resposta', 'status_prazo_resposta']
Salvo: Data Layer/silver/tb_mapa_cidadania_ra_ano_silver.csv | linhas=99 colunas=15


/tmp/ipykernel_472832/4007278850.py:274: DtypeWarning: Columns (0: e03_1_1, 1: e03_2_1) have mixed types. Specify dtype option on import or set low_memory=False.
  resumo=pd.DataFrame({'arquivo':arquivos_esperados,'linhas':[len(pd.read_csv(SILVER_DIR/nome)) for nome in arquivos_esperados]})


/tmp/ipykernel_472832/4007278850.py:274: DtypeWarning: Columns (0: dt_pedido, 1: dt_data_pedido, 2: dt_inicio_prazo, 3: dt_termino_prazo, 4: dt_max_prazo_resposta, 5: respostas_dt_resposta, 6: ano_mes) have mixed types. Specify dtype option on import or set low_memory=False.
  resumo=pd.DataFrame({'arquivo':arquivos_esperados,'linhas':[len(pd.read_csv(SILVER_DIR/nome)) for nome in arquivos_esperados]})


,arquivo,linhas
0,tb_pdad_moradores_silver.csv,69542
1,tb_pdad_domicilios_silver.csv,24845
2,tb_lai_pedidos_silver.csv,70946
3,tb_lai_recursos_silver.csv,3998
4,tb_lai_satisfacao_silver.csv,7369
5,tb_populacao_ra_ano_silver.csv,99
6,tb_mapa_cidadania_ra_ano_silver.csv,99
7,tb_lai_orgao_ano_silver.csv,288
